# 그림 10 다시 그리기 (범례 용어 수정)

원고는 보정한 90% 구간의 하한을 **LB90**이라고 부르는데, 기존 그림 10의 범례는 `PI-BDP q05 (calibrated)`로 되어 있습니다. 이 노트북은 **새 계산 없이** 정비 시뮬레이션의 저장된 결과로 그림만 다시 그립니다.

- (a) ρ 민감도: `posthoc_maint/maint_table.csv`의 값을 그대로 씁니다.
- (b) 고장률 대 미사용 수명: 저장된 test 예측으로 같은 시뮬레이터를 다시 돌립니다(학습 없음, 1분 이내).
- 그리기 전에 ρ = 10의 값이 원고 표 5와 같은지 확인합니다.

**결과:** 드라이브 `posthoc_maint/Figure_10.png`, `Figure_10.pdf`. 두 파일을 올려 주세요. 런타임은 CPU면 됩니다.

In [ ]:
import os, sys, json, glob
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
OUT = f'{ROOT}/posthoc_maint'
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
L_LEAD, DT, RHOS = 5, 1, [2, 5, 10, 20, 50]
TAU0 = L_LEAD + DT
K = ['subset', 'unit', 'cycle']

RES = pd.read_csv(f'{OUT}/maint_table.csv')
print(RES.columns.tolist())
# 원고 표 5와 같은 값인지 확인 (rho = 10, 조정 없음)
chk = RES[(RES.rho == 10) & (~RES.group.str.startswith('tuned'))].set_index('policy').downtime.round(1)
print(chk)
assert abs(chk['PI-BDP q05 (calibrated)'] - 51.9) < 0.06 and abs(chk['TBM (Weibull optimal age)'] - 42.0) < 0.06, '저장된 결과가 원고와 다릅니다'
print('표 5 값과 일치')

In [ ]:
# === (b)용: 저장된 test 예측과 같은 시뮬레이터 (학습 없음) ===
def frame(df, stat):
    d = df[K].copy(); d['stat'] = stat; d['eol'] = (df.cycle + df.true).values
    return d.sort_values(K).reset_index(drop=True)

def simulate(d, tau):
    out = []
    for (s, u), g in d.groupby(['subset', 'unit'], sort=False):
        eol = int(g.eol.iloc[0]); hit = g.cycle.values[g.stat.values <= tau]
        if len(hit) and hit[0] + L_LEAD < eol: out.append((s, u, eol, hit[0] + L_LEAD, 0))
        else: out.append((s, u, eol, eol, 1))
    return pd.DataFrame(out, columns=['subset', 'unit', 'eol', 'used', 'fail'])

v0t = pd.concat([pd.read_csv(f'{ROOT}/phase3/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
TEST = {}
for n in ['GB', 'LSTM']:
    d = pd.read_csv(f'{ROOT}/phase3/test_{n}.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean()
    TEST[n] = frame(d, d.pred.values)
rc = pd.concat([pd.read_csv(p) for p in glob.glob(f'{ROOT}/posthoc_cnn/pred_*_seed*.csv.gz')], ignore_index=True)
rc = rc.groupby(K + ['seed'], as_index=False)[['pred', 'true']].mean().groupby(K, as_index=False)[['pred', 'true']].mean()
TEST['CNN (lit.-style)'] = frame(rc, rc.pred.values)
print('준비 완료:', {k: v.groupby(['subset', 'unit']).ngroups for k, v in TEST.items()})

In [ ]:
# === 그림 10: 기존 그림과 같은 설정, 범례만 원고 용어로 ===
import matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.5, 'legend.frameon': False, 'savefig.dpi': 300})
COL = {'PI-BDP q05 (calibrated)': '#2a78d6', 'PI-BDP q05 (raw)': '#86b6ef', 'GB': '#eb6834', 'LSTM': '#1baf7a',
       'CNN (lit.-style)': '#eda100', 'TBM (Weibull optimal age)': '#52514e', 'RF': '#e87ba4'}
LABEL = {'PI-BDP q05 (calibrated)': 'PI-BDP LB90 (calibrated)', 'PI-BDP q05 (raw)': 'PI-BDP q05 (uncalibrated)',
         'TBM (Weibull optimal age)': 'TBM (optimal age)'}
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0))
ax = axs[0]
for pol, c in COL.items():
    g = RES[(RES.policy == pol) & (~RES.group.str.startswith('tuned'))].sort_values('rho')
    ax.plot(g.rho, g.downtime, '-o', color=c, ms=3.5, lw=2.0 if 'calibrated' in pol else 1.2, label=LABEL.get(pol, pol))
ax.set_xscale('log'); ax.set_xticks(RHOS); ax.set_xticklabels(RHOS)
ax.set_xlabel('Unplanned / planned downtime ratio ρ'); ax.set_ylabel('Downtime per 1,000 cycles')
ax.set_title('(a) Untuned policies (τ = L + Δt)', loc='left'); ax.legend(fontsize=6.5)
ax = axs[1]
qcols = ['q025', 'q050', 'q100', 'q250', 'q500']
pts = [simulate(frame(v0t, v0t[c].values), TAU0) for c in qcols]
ax.plot([((r.eol - r.used) / r.eol).mean() * 100 for r in pts], [r.fail.mean() * 100 for r in pts], '-o', color='#2a78d6', ms=3.5,
        label='PI-BDP, quantile q varied (uncalibrated)')
for n in ['GB', 'LSTM', 'CNN (lit.-style)']:
    ps = [simulate(TEST[n], t) for t in range(0, 31, 2)]
    ax.plot([((r.eol - r.used) / r.eol).mean() * 100 for r in ps], [r.fail.mean() * 100 for r in ps], '-o', color=COL[n], ms=2.5, lw=1.0,
            label=f'{n}, threshold τ varied')
ax.set_xlabel('Mean unused life (%)'); ax.set_ylabel('Unplanned failures (%)')
ax.set_title('(b) Trade-off (descriptive)', loc='left'); ax.legend(fontsize=6.5)
fig.tight_layout()
fig.savefig(f'{OUT}/Figure_10.png', bbox_inches='tight'); fig.savefig(f'{OUT}/Figure_10.pdf', bbox_inches='tight')
plt.show(); print('저장:', f'{OUT}/Figure_10.png', f'{OUT}/Figure_10.pdf')